# Building a Simple LLM with PyTorch: Fine-tuning for Sentiment Analysis

In this notebook, we'll fine-tune our pre-trained language model for sentiment analysis using the IMDB movie reviews dataset. We'll cover:

1. Loading our pre-trained model
2. Preparing the IMDB dataset
3. Implementing fine-tuning
4. Evaluating the fine-tuned model
5. Using the model for sentiment analysis

Let's begin!

## 1. Setup and Model Loading

First, let's import our dependencies and load our pre-trained model:

In [ ]:
import sys
from pathlib import Path

# Add project root to Python path
project_root = Path.cwd().parent
sys.path.append(str(project_root))

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
import numpy as np
from tqdm import tqdm

from src.model import create_model, LLM
from src.data import DataModule
from src.inference_pipeline import create_inference_pipeline

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load pre-trained model
save_dir = Path("saved_model")
checkpoint = torch.load(save_dir / 'model.pt', map_location=device)

# Create model with saved config
model = create_model(
    vocab_size=checkpoint['config']['vocab_size'],
    hidden_size=checkpoint['config']['hidden_size'],
    num_layers=checkpoint['config']['num_hidden_layers'],
    num_heads=checkpoint['config']['num_attention_heads']
)

# Load state dict
model.load_state_dict(checkpoint['model_state_dict'])
model = model.to(device)

# Initialize data module to get tokenizer
data_module = DataModule()
data_module.prepare_data()

## 2. Preparing IMDB Dataset

Let's create a custom dataset class for IMDB reviews:

In [ ]:
class IMDBDataset(Dataset):
    """Dataset for IMDB movie reviews."""
    
    def __init__(self, reviews, labels, tokenizer, max_length=512):
        self.reviews = reviews
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
        
    def __len__(self):
        return len(self.reviews)
    
    def __getitem__(self, idx):
        review = self.reviews[idx]
        label = self.labels[idx]
        
        # Prepare input text
        input_text = f"Review: {review}\nSentiment:"
        
        # Tokenize
        encoding = self.tokenizer.encode(input_text)
        input_ids = encoding.ids[:self.max_length]
        
        # Pad if necessary
        if len(input_ids) < self.max_length:
            input_ids = input_ids + [self.tokenizer.token_to_id("[PAD]")] * (self.max_length - len(input_ids))
        
        return {
            "input_ids": torch.tensor(input_ids),
            "attention_mask": torch.ones(self.max_length),
            "labels": torch.tensor(label, dtype=torch.long)
        }

# Load IMDB dataset
print("Loading IMDB dataset...")
imdb_dataset = load_dataset("imdb")

# Create train and validation datasets
train_dataset = IMDBDataset(
    imdb_dataset["train"]["text"],
    imdb_dataset["train"]["label"],
    data_module.tokenizer
)

val_dataset = IMDBDataset(
    imdb_dataset["test"]["text"],
    imdb_dataset["test"]["label"],
    data_module.tokenizer
)

# Create data loaders
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16)

print(f"Training samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")

## 3. Implementing Fine-tuning

Let's create a sentiment classifier by adding a classification head to our pre-trained model:

In [ ]:
class SentimentClassifier(nn.Module):
    """Sentiment classifier using pre-trained LLM."""
    
    def __init__(self, pretrained_model: LLM, num_classes: int = 2):
        super().__init__()
        self.pretrained_model = pretrained_model
        
        # Freeze the pre-trained model parameters
        for param in self.pretrained_model.parameters():
            param.requires_grad = False
            
        # Add classification head
        self.classifier = nn.Sequential(
            nn.Linear(pretrained_model.config.hidden_size, 256),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(256, num_classes)
        )
    
    def forward(self, input_ids, attention_mask):
        # Get the hidden states from pre-trained model
        outputs = self.pretrained_model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        
        # Use the last hidden state of the first token ([CLS])
        hidden_state = outputs[0][:, 0, :]
        
        # Classify
        logits = self.classifier(hidden_state)
        return logits

# Create sentiment classifier
classifier = SentimentClassifier(model).to(device)

# Training parameters
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(classifier.parameters(), lr=2e-5)
num_epochs = 3

Now let's implement the training loop:

In [ ]:
def train_epoch(model, dataloader, criterion, optimizer, device):
    """Train for one epoch."""
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for batch in tqdm(dataloader, desc="Training"):
        # Move batch to device
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        
        # Forward pass
        optimizer.zero_grad()
        outputs = model(input_ids, attention_mask)
        loss = criterion(outputs, labels)
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        # Track metrics
        total_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    
    return total_loss / len(dataloader), correct / total

def evaluate(model, dataloader, criterion, device):
    """Evaluate model."""
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Evaluating"):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
            
            outputs = model(input_ids, attention_mask)
            loss = criterion(outputs, labels)
            
            total_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    return total_loss / len(dataloader), correct / total

# Training loop
print("Starting fine-tuning...")
for epoch in range(num_epochs):
    print(f"\nEpoch {epoch + 1}/{num_epochs}")
    
    # Train
    train_loss, train_acc = train_epoch(
        classifier,
        train_loader,
        criterion,
        optimizer,
        device
    )
    
    # Evaluate
    val_loss, val_acc = evaluate(
        classifier,
        val_loader,
        criterion,
        device
    )
    
    print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
    print(f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")

# Save fine-tuned model
torch.save({
    'model_state_dict': classifier.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
}, 'saved_model/sentiment_classifier.pt')

## 4. Using the Fine-tuned Model

Let's create a simple interface for sentiment analysis:

In [ ]:
def analyze_sentiment(text: str, model, tokenizer, device):
    """Analyze sentiment of given text."""
    model.eval()
    
    # Prepare input
    input_text = f"Review: {text}\nSentiment:"
    encoding = tokenizer.encode(input_text)
    input_ids = torch.tensor([encoding.ids[:512]]).to(device)
    attention_mask = torch.ones_like(input_ids).to(device)
    
    # Get prediction
    with torch.no_grad():
        outputs = model(input_ids, attention_mask)
        probabilities = torch.softmax(outputs, dim=1)
        prediction = torch.argmax(probabilities, dim=1).item()
        confidence = probabilities[0][prediction].item()
    
    return {
        'sentiment': 'Positive' if prediction == 1 else 'Negative',
        'confidence': confidence
    }

# Test the sentiment analyzer
test_reviews = [
    "This movie was absolutely fantastic! The acting was superb and the plot kept me engaged throughout.",
    "I was really disappointed with this film. The story was confusing and the characters were poorly developed.",
    "While it had some good moments, overall the movie was just average.",
    "An absolute masterpiece! One of the best films I've seen in years."
]

print("Analyzing test reviews...\n")
for review in test_reviews:
    result = analyze_sentiment(review, classifier, data_module.tokenizer, device)
    print(f"Review: {review}")
    print(f"Sentiment: {result['sentiment']} (Confidence: {result['confidence']:.2%})\n")

## 5. Comparing Pre-trained and Fine-tuned Models

Let's compare how our model performs before and after fine-tuning:

In [ ]:
# Create inference pipeline for pre-trained model
pipeline = create_inference_pipeline(
    model=model,
    tokenizer=data_module.tokenizer,
    device=device
)

def compare_models(review):
    print(f"Review: {review}")
    print("\nPre-trained model completion:")
    completion = pipeline.generate(
        prompt=f"Review: {review}\nSentiment:",
        max_length=50,
        temperature=0.7,
        top_p=0.9
    )[0]
    print(completion)
    
    print("\nFine-tuned model analysis:")
    result = analyze_sentiment(review, classifier, data_module.tokenizer, device)
    print(f"Sentiment: {result['sentiment']} (Confidence: {result['confidence']:.2%})")
    print("-" * 80)

# Test comparison
test_reviews = [
    "The special effects were amazing, but the story felt hollow and the characters were one-dimensional.",
    "This film is a perfect example of how to blend action, drama, and humor into a compelling narrative."
]

for review in test_reviews:
    compare_models(review)

## 6. Best Practices for Fine-tuning

Here are some key takeaways for fine-tuning language models:

1. **Data Preparation**:
   - Clean and preprocess your data carefully
   - Ensure balanced class distribution
   - Use appropriate text formatting

2. **Model Architecture**:
   - Start by freezing the pre-trained weights
   - Add task-specific layers
   - Consider gradual unfreezing

3. **Training Process**:
   - Use a small learning rate
   - Monitor validation metrics
   - Implement early stopping

4. **Evaluation**:
   - Use appropriate metrics
   - Test on diverse examples
   - Compare with baseline models

## Next Steps

We've successfully:
1. Fine-tuned our pre-trained model for sentiment analysis
2. Created a simple sentiment analysis interface
3. Compared pre-trained and fine-tuned performance
4. Learned best practices for fine-tuning

You can now adapt this approach to fine-tune the model for other specific tasks!